## Importing Libraries

In [1]:
import re
import os
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from typing import Optional
from IPython.display import display

tqdm.pandas()

In [2]:
interim_file_path = 'Data/interim'
os.makedirs(interim_file_path, exist_ok = True)

## Data Cleaning

In [3]:
district_mapping = {'Mulugu' : ['Mulug'],
                    'Hanumakonda' : ['Hanumakkonda'],
                    'Jangaon' : ['Jangoan'],
                    'Mahabubnagar' : ['Mahbubnagar'],
                    'Warangal' : ['Warangal - Rural', 'Warangal Rural', 'Warangal (R)',
                                  'Warangal - Urban', 'Warangal Urban', 'Warangal (U)'],
                    'Bhadradri Kothagudem' : ['BHADRADRI KOTHAGUDEM', 'Bhadradri-Kothagudem'],
                    'Komaram Bheem Asifabad' : ['Asifabad', 'Komaram Bheem-Asifabad',
                                                'Kumuram Bheem', 'Kumuram Bheem - Asifabad'],
                    'Jayashankar Bhupalpally' : ['Jayashankar', 'Jayashankar-Bhupalpally',
                                                 'Bhupalpally', 'Jayashankar Bhoopalpally'],
                    'Yadadri Bhuvanagiri' : ['Yadadri', 'Yadadri-Bhongir'],
                    'Rajanna Sircilla' : ['Sircilla', 'Rajanna-Siricilla'],
                    'Ranga Reddy' : ['Rangareddy'],
                    'Jogulamba Gadwal' : ['Gadwal', 'Jogulamba-Gadwal'],
                    'Medchal-Malkajgiri' : ['Medchal']}

### 1. Weather Data

**Column Details**

- `District` : Name of the district
- `Date` : Date of record
- `RainFall` : Average Cumulative Rainfall in mm
- `Min_Temp` : Average Minimum Temperature in celcius
- `Max_Temp` : Average Maximum Temperature in celcius
- `Min_Humidity` : AverageMinimum Humidity %
- `Max_Humidity` : Average Maximum Humidity %

In [4]:
weather_mapping = {'District' : ['district'],
                   'Mandal' : ['mandal'],
                   'Min Humidity (%)' : ['humidity_min', 'Humidity Min (%)', 'humidity_min (%)'],
                   'Max Humidity (%)' : ['humidity_max', 'Humidity Max (%)', 'Humidity_max (%)',
                                         'humidity_max (%)'],
                   'Min Wind Speed (Kmph)' : ['wind_speed_min', 'Wind Speed Min (Kmph)',
                                              'wind_speed_min (Kmph)'],
                   'Max Wind Speed (Kmph)' : ['wind_speed_max', 'Wind Speed Max (Kmph)',
                                              'wind_speed_max (Kmph)'],
                   'Min Temp (°C)' : ['temp_min (⁰C)', 'Temp Max (°C)', 'temp_min'],
                   'Max Temp (°C)' : ['temp_max (⁰C)', 'Temp Min (°C)', 'temp_max'],
                   'Rain (mm)' : ['Rainfall (mm)', 'rain', 'cumm_rainfall'],
                   'Date' : ['odate', 'date']}

weather_selected_columns = ['District', 'Date', 'Rain (mm)', 'Min Temp (°C)',
                            'Max Temp (°C)', 'Min Humidity (%)', 'Max Humidity (%)']

#### 1.1 Combining all available Weather Data

In [5]:
Weather_DF = None

for file in tqdm(glob('Data/raw/Weather_Data/*')):
    try:
        temp = pd.read_csv(file).rename(columns = {value:
                        key for key, values in weather_mapping.items() for value in values})
    except:
        temp = pd.read_excel(file).rename(columns = {value:
                        key for key, values in weather_mapping.items() for value in values})
    finally:
        if isinstance(temp.Date.values[0], str):
            if re.search(r'\d{2}-[A-Za-z]{3}-\d{2}', temp.Date.values[0]): # 01-Sep-24
                temp.Date = pd.to_datetime(temp.Date, format = '%d-%b-%y')
            elif re.search(r'\d{2}\/\d{2}\/\d{2}', temp.Date.values[0]): # 01/01/18
                temp.Date = pd.to_datetime(temp.Date, format = '%d/%m/%y')
            else:
                temp.Date = pd.to_datetime(temp.Date, format='mixed')
        if not isinstance(Weather_DF, pd.DataFrame):
            Weather_DF = temp[weather_selected_columns].copy(deep = True)
        else:
            Weather_DF = pd.concat([Weather_DF, temp[weather_selected_columns]])

Weather_DF.District = Weather_DF.District.replace({value:
                        key for key, values in district_mapping.items() for value in values})

Weather_DF.columns = Weather_DF.columns.str.replace(r'\(.*\)', '', regex = True).str.strip()
Weather_DF.columns = Weather_DF.columns.str.replace(r'\s', '_', regex = True)
Weather_DF.rename(columns = {'Rain' : 'RainFall'}, inplace = True)

print(f'Number of Districts in Weather Dataset\t\t: {Weather_DF.District.nunique()}')
print(f'Number of data points in the  Weather Dataset\t: {Weather_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t: {sum(Weather_DF.isnull().sum())}\n')

Weather_DF.head()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 56/56 [00:08<00:00,  6.67it/s]


Number of Districts in Weather Dataset		: 33
Number of data points in the  Weather Dataset	: 1392219
Number of missing values in the dataset		: 16



,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Adilabad,2024-09-01,88.0,23.6,31.7,79.3,100.0
1,Adilabad,2024-09-02,17.1,24.7,31.0,81.5,100.0
2,Adilabad,2024-09-03,33.4,24.9,33.1,80.8,100.0
3,Adilabad,2024-09-04,3.4,23.9,27.9,91.0,100.0
4,Adilabad,2024-09-05,2.7,23.6,30.0,85.9,100.0


#### 1.2 Handling Date-Time data and defining time period

- Start point : `01-01-2019`
- End point   : `31-12-2024`

In [6]:
Weather_DF.Date = pd.to_datetime(Weather_DF.Date)
Weather_DF = Weather_DF.query('2019 <= Date < 2025').sort_values('Date').reset_index(drop = True)

print(f'Number of Districts in Weather Dataset\t\t: {Weather_DF.District.nunique()}')
print(f'Number of data points in the  Weather Dataset\t: {Weather_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t: {sum(Weather_DF.isnull().sum())}\n')

Weather_DF.head()

Number of Districts in Weather Dataset		: 33
Number of data points in the  Weather Dataset	: 1356111
Number of missing values in the dataset		: 16



,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Rajanna Sircilla,2019-01-01,0.0,6.7,25.8,22.9,55.9
1,Khammam,2019-01-01,0.0,15.5,30.5,35.1,90.0
2,Jayashankar Bhupalpally,2019-01-01,0.0,10.6,26.6,25.6,90.0
3,Nagarkurnool,2019-01-01,0.0,12.6,30.3,17.7,75.4
4,Vikarabad,2019-01-01,0.0,11.4,32.9,18.4,64.0


#### 1.3 Handling Missing Values

In [7]:
print('Missing Values in the Weather Dataset')
Weather_DF.isnull().sum()

Missing Values in the Weather Dataset


District         0
Date             0
RainFall         0
Min_Temp         0
Max_Temp         0
Min_Humidity    12
Max_Humidity     4
dtype: int64

In [8]:
# https://pandas.pydata.org/docs/user_guide/style.html#Acting-on-Data
def style_negative(v: str, props: str = '') -> Optional[str]:
    return props if v < 0 else None

def style_nan(v: float, props: str = '') -> Optional[str]:
    if isinstance(v, str):
        return None
    return props if np.isnan(v) else None

In [9]:
missing_valueDF = Weather_DF[Weather_DF['Min_Humidity'].isna() | Weather_DF['Max_Humidity'].isna()]
missing_valueDF.reset_index(drop = True, inplace = True)
subset = ['Min_Humidity', 'Max_Humidity']
display(missing_valueDF.style.map(style_nan, props = 'color:red;', subset = subset))
print('>>> Since, these might be due to entry error, removing these rows fully')
Weather_DF.dropna(inplace = True)
print(f'>>> Number of missing values in the revised dataset : {sum(Weather_DF.isnull().sum())}')

,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Nagarkurnool,2019-03-07 00:00:00,0.000000,17.900000,34.400000,nan,nan
1,Karimnagar,2019-12-14 00:00:00,0.000000,20.200000,31.000000,nan,100.000000
2,Vikarabad,2019-12-19 00:00:00,0.000000,14.700000,30.600000,nan,96.600000
3,Vikarabad,2019-12-19 00:00:00,0.000000,14.566667,30.433333,nan,97.966667
4,Vikarabad,2019-12-20 00:00:00,0.000000,15.200000,30.700000,nan,99.200000
5,Vikarabad,2019-12-20 00:00:00,0.000000,15.200000,31.000000,nan,96.666667
6,Rajanna Sircilla,2019-12-20 00:00:00,0.000000,13.700000,30.500000,nan,99.800000
7,Karimnagar,2019-12-20 00:00:00,0.000000,15.900000,30.200000,nan,100.000000
8,Khammam,2020-10-30 00:00:00,0.000000,33.700000,22.400000,nan,100.000000
9,Khammam,2022-05-01 00:00:00,0.000000,27.200000,42.200000,23.000000,nan


>>> Since, these might be due to entry error, removing these rows fully
>>> Number of missing values in the revised dataset : 0


In [10]:
Weather_DF.describe(include = [np.number]).style.map(style_negative, props = 'color:red;')

,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
count,1356096.000000,1356096.000000,1356096.000000,1356096.000000,1356096.000000
mean,3.049469,22.459836,33.482874,47.171182,88.364400
std,11.232039,4.546942,5.018502,21.025809,15.607864
min,0.000000,0.000000,-1.000000,-1.000000,-1.000000
25%,0.000000,19.700000,30.900000,30.700000,82.900000
50%,0.000000,23.100000,33.200000,46.000000,94.400000
75%,0.000000,25.100000,36.700000,62.800000,99.900000
max,618.500000,37.400000,47.900000,100.000000,100.000000


`Max_Temp`, `Min_Humidity`, and `Max_Humidity` can't be **negative**, so need to check those.

In [11]:
neg_Max_Temp = Weather_DF.query('Max_Temp == -1').reset_index(drop = True)
neg_Min_Humidity = Weather_DF.query('Min_Humidity == -1').reset_index(drop = True)
neg_Max_Humidity = Weather_DF.query('Max_Humidity == -1').reset_index(drop = True)

print(f"Number of occations '-1' appeared in 'Max_Temp' column     : {neg_Max_Temp.shape[0]}")
print(f"Number of occations '-1' appeared in 'Min_Humidity' column : {neg_Min_Humidity.shape[0]}")
print(f"Number of occations '-1' appeared in 'Max_Humidity' column : {neg_Max_Humidity.shape[0]}\n")

subset = ['Max_Temp', 'Min_Humidity', 'Max_Humidity']
# display(neg_Max_Temp.head().style.map(style_negative, props = 'color:red;', subset = subset))
display(neg_Min_Humidity.head().style.map(style_negative, props = 'color:red;', subset = subset))
# display(neg_Max_Humidity.head().style.map(style_negative, props = 'color:red;', subset = subset))

Number of occations '-1' appeared in 'Max_Temp' column     : 3
Number of occations '-1' appeared in 'Min_Humidity' column : 32
Number of occations '-1' appeared in 'Max_Humidity' column : 10



,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Nalgonda,2021-03-21 00:00:00,0.000000,19.800000,32.300000,-1.000000,95.500000
1,Khammam,2021-07-29 00:00:00,0.000000,28.800000,34.000000,-1.000000,79.500000
2,Khammam,2021-07-29 00:00:00,0.000000,28.800000,34.600000,-1.000000,79.500000
3,Peddapalli,2022-03-19 00:00:00,0.000000,22.900000,37.300000,-1.000000,100.000000
4,Peddapalli,2022-03-19 00:00:00,0.000000,24.800000,36.200000,-1.000000,99.900000


In [12]:
peddapalli_Weather = Weather_DF.query('District == "Peddapalli"').reset_index(drop = True)
peddapalli_Weather.query('Min_Humidity == -1')
peddapalli_Weather.iloc[[17696, 17697, 17698, 17699, 17700, 17701, 17702]].style.map(
                                    style_negative, props = 'color:red;', subset = ['Min_Humidity'])

,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
17696,Peddapalli,2022-03-19 00:00:00,0.000000,22.700000,37.800000,39.300000,89.300000
17697,Peddapalli,2022-03-19 00:00:00,0.000000,22.600000,37.500000,43.400000,96.500000
17698,Peddapalli,2022-03-19 00:00:00,0.000000,22.900000,37.300000,-1.000000,100.000000
17699,Peddapalli,2022-03-19 00:00:00,0.000000,23.700000,36.300000,55.000000,99.500000
17700,Peddapalli,2022-03-19 00:00:00,0.000000,24.800000,36.200000,-1.000000,99.900000
17701,Peddapalli,2022-03-19 00:00:00,0.000000,24.100000,36.400000,41.100000,88.000000
17702,Peddapalli,2022-03-19 00:00:00,0.000000,23.900000,34.100000,53.100000,87.700000


In [13]:
print('Since, again these might be due to entry error, removing these rows fully')
Weather_DF = Weather_DF.query('Max_Temp != -1 and Min_Humidity != -1 and Max_Humidity != -1')
Weather_DF = Weather_DF.sort_values(['Date', 'District']).reset_index(drop = True)

print(f'Number of datapoints after removing missing values : {Weather_DF.shape[0]}\n')
Weather_DF.head()

Since, again these might be due to entry error, removing these rows fully
Number of datapoints after removing missing values : 1356051



,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Adilabad,2019-01-01,0.0,6.2,27.6,17.2,88.0
1,Adilabad,2019-01-01,0.0,6.0,26.6,21.6,96.4
2,Adilabad,2019-01-01,0.0,5.7,27.4,18.8,75.1
3,Adilabad,2019-01-01,0.0,8.1,28.1,20.5,64.0
4,Adilabad,2019-01-01,0.0,9.5,25.6,20.0,66.0


#### 1.4 Type-Casting to correct datatype

In [14]:
# Type-Casting to correct datatype

Weather_DF.District = Weather_DF.District.astype('string')

display(Weather_DF.info())
Weather_DF.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1356051 entries, 0 to 1356050
Data columns (total 7 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   District      1356051 non-null  string        
 1   Date          1356051 non-null  datetime64[ns]
 2   RainFall      1356051 non-null  float64       
 3   Min_Temp      1356051 non-null  float64       
 4   Max_Temp      1356051 non-null  float64       
 5   Min_Humidity  1356051 non-null  float64       
 6   Max_Humidity  1356051 non-null  float64       
dtypes: datetime64[ns](1), float64(5), string(1)
memory usage: 72.4 MB


None

,District,Date,RainFall,Min_Temp,Max_Temp,Min_Humidity,Max_Humidity
0,Adilabad,2019-01-01,0.0,6.2,27.6,17.2,88.0
1,Adilabad,2019-01-01,0.0,6.0,26.6,21.6,96.4
2,Adilabad,2019-01-01,0.0,5.7,27.4,18.8,75.1
3,Adilabad,2019-01-01,0.0,8.1,28.1,20.5,64.0
4,Adilabad,2019-01-01,0.0,9.5,25.6,20.0,66.0


#### 1.5 Saving Weather Dataset

In [15]:
# Saving DataFrame to Parquet

weather_path = f'{interim_file_path}/Weather_Data.parquet'
print(f'Saving cleaned Weather Dataset to "{weather_path}"')
if not os.path.isfile(weather_path):
    Weather_DF.to_parquet(weather_path, engine = 'pyarrow')

Saving cleaned Weather Dataset to "Data/interim/Weather_Data.parquet"


### 2. Registration and Stamps Data (Non-Agriculture)

**Column Details**

- `Date` : Data of documents registered
- `Dist_Name` : District Name
- `Documents_Registered_Cnt` : Total Documents Registered Count
- `Documents_Registered_Rev` : Total Documents Registered Revenue
- `Estamps_Challans_Cnt` : E-stamps challans count
- `Estamps_Challans_Rev` : E-stamps challans revenue
- `Slot_Booking_Cnt` : Total online Slot Booking Count

#### 2.1 Combining all available Registration Data

In [16]:
registration_selected_columns = ['Date', 'Dist_Name', 'Documents_Registered_Cnt',
                                 'Documents_Registered_Rev', 'Estamps_Challans_Cnt',
                                 'Estamps_Challans_Rev', 'Slot_Booking_Cnt']

In [17]:
registration_DF = None

for file in tqdm(glob('Data/raw/Registration_Stamps/*')):
    try:
        temp = pd.read_csv(file)
    except:
        print(f"Unabale to read the file -> {file}")
    finally:
        temp.stats_date = pd.to_datetime(temp.stats_date, format = '%d-%m-%Y')
        if not isinstance(registration_DF, pd.DataFrame):
            registration_DF = temp.copy(deep = True)
        else:
            registration_DF = pd.concat([registration_DF, temp])

# Renaming columns and removing white spaces
registration_DF.columns = [col.replace('_', ' ').title().replace(' ', '_')
                                                            for col in registration_DF.columns]

# Unifying Values
registration_DF.Dist_Name = registration_DF.Dist_Name.str.title()
registration_DF.Sro_Name = registration_DF.Sro_Name.str.title()
registration_DF.Dist_Name = registration_DF.Dist_Name.replace({value:
                            key for key, values in district_mapping.items() for value in values})
registration_DF.rename(columns = {'Stats_Date' : 'Date'}, inplace = True)

# Selecting needed columns only
registration_DF = registration_DF[registration_selected_columns]

print(f'Number of Districts in Registration Dataset\t\t: {registration_DF.Dist_Name.nunique()}')
print(f'Number of data points in the  Registration Dataset\t: {registration_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t\t: {sum(registration_DF.isnull().sum())}\n')

registration_DF.head()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 71/71 [00:00<00:00, 97.22it/s]


Number of Districts in Registration Dataset		: 33
Number of data points in the  Registration Dataset	: 306677
Number of missing values in the dataset			: 0



,Date,Dist_Name,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
0,2021-07-01,Nagarkurnool,20,161130,11,47720,0
1,2021-07-01,Jogulamba Gadwal,6,41475,4,14150,0
2,2021-07-01,Wanaparthy,8,77685,10,62835,0
3,2021-07-01,Jogulamba Gadwal,64,2114670,68,1130250,0
4,2021-07-01,Mahabubnagar,47,754197,41,838653,1


#### 2.2 Handling Date-Time data and defining time period

- Start point : `01-01-2019`
- End point   : `31-12-2024`

In [18]:
registration_DF.Date = pd.to_datetime(registration_DF.Date)
registration_DF = registration_DF.query('2019 <= Date < 2025')
registration_DF = registration_DF.sort_values('Date').reset_index(drop = True)

print(f'Number of Districts in Registration Dataset\t\t: {registration_DF.Dist_Name.nunique()}')
print(f'Number of data points in the  Registration Dataset\t: {registration_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t\t: {sum(registration_DF.isnull().sum())}\n')

registration_DF.head()

Number of Districts in Registration Dataset		: 33
Number of data points in the  Registration Dataset	: 306677
Number of missing values in the dataset			: 0



,Date,Dist_Name,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
0,2019-01-01,Bhadradri Kothagudem,0,0,0,0,0
1,2019-01-01,Ranga Reddy,0,0,0,0,1
2,2019-01-01,Siddipet,0,0,0,0,7
3,2019-01-01,Rajanna Sircilla,0,0,0,0,0
4,2019-01-01,Karimnagar,0,0,0,0,0


#### 2.3 Handling Missing Values

In [19]:
print(f'Number of missing values in the dataset : {sum(registration_DF.isnull().sum())}\n')
registration_DF.isnull().sum()

Number of missing values in the dataset : 0



Date                        0
Dist_Name                   0
Documents_Registered_Cnt    0
Documents_Registered_Rev    0
Estamps_Challans_Cnt        0
Estamps_Challans_Rev        0
Slot_Booking_Cnt            0
dtype: int64

In [20]:
registration_DF.describe(include = np.number)

,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
count,306677.000000,3.066770e+05,306677.000000,3.066770e+05,306677.000000
mean,23.978104,1.655724e+06,16.490112,1.364361e+06,2.054060
std,33.223862,6.270801e+06,27.222562,5.690234e+06,8.780325
min,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
50%,13.000000,2.152500e+05,2.000000,4.020000e+04,0.000000
75%,36.000000,1.157000e+06,24.000000,7.962400e+05,0.000000
max,1235.000000,5.351640e+08,549.000000,5.198990e+08,473.000000


Except for `Dist_Name`, all other columns are numerical, and each should have a minimum value greater than or equal to **zero**, which is satisfied here. Therefore, there are no missing values.

#### 2.4 Type-Casting to correct datatype

In [21]:
# Type-Casting to correct datatype

registration_DF.Dist_Name = registration_DF.Dist_Name.astype('string')

display(registration_DF.info())
registration_DF.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306677 entries, 0 to 306676
Data columns (total 7 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Date                      306677 non-null  datetime64[ns]
 1   Dist_Name                 306677 non-null  string        
 2   Documents_Registered_Cnt  306677 non-null  int64         
 3   Documents_Registered_Rev  306677 non-null  int64         
 4   Estamps_Challans_Cnt      306677 non-null  int64         
 5   Estamps_Challans_Rev      306677 non-null  int64         
 6   Slot_Booking_Cnt          306677 non-null  int64         
dtypes: datetime64[ns](1), int64(5), string(1)
memory usage: 16.4 MB


None

,Date,Dist_Name,Documents_Registered_Cnt,Documents_Registered_Rev,Estamps_Challans_Cnt,Estamps_Challans_Rev,Slot_Booking_Cnt
0,2019-01-01,Bhadradri Kothagudem,0,0,0,0,0
1,2019-01-01,Ranga Reddy,0,0,0,0,1
2,2019-01-01,Siddipet,0,0,0,0,7
3,2019-01-01,Rajanna Sircilla,0,0,0,0,0
4,2019-01-01,Karimnagar,0,0,0,0,0


#### 2.5 Saving Registration Dataset

In [22]:
# Saving DataFrame to Parquet

registration_path = f'{interim_file_path}/Registration_Data.parquet'
print(f'Saving cleaned Registration Dataset to "{registration_path}"')
if not os.path.isfile(registration_path):
    registration_DF.to_parquet(registration_path, engine = 'pyarrow')

Saving cleaned Registration Dataset to "Data/interim/Registration_Data.parquet"


### 3. Telangana Industries TS-iPASS Data

**Column Details**

- `District_Name` : Names of district
- `Name_Of_The_Unit` : Name and address of the unit
- `Line_Of_Activity` : Line of activiity of the organization
- `Sector` : Sector which the industry belongs to
- `Investment` : Investment amount in Millions
- `Number_Of_Employees` : Number of employees in the organization
- `Application_Date` : Date of the application registered in the government for approval
- `Approval_Date` : Date of approval for the application submitted
- `Progress_Of_Implementation` : Indicated status of the businesses
- `Social_Status` : Category of the applicant

In [23]:
ipass_column_names = {'district' : 'district_name',
                      'mandal' : 'mandal_name',
                      'village' : 'village_name',
                      'unit_name' : 'name_of_the_unit',
                      'in_online' : 'is_online'}

registration_selected_columns = ['District_Name', 'Name_Of_The_Unit', 'Line_Of_Activity', 'Sector',
                                 'Investment', 'Number_Of_Employees',  'Application_Date',
                                 'Approval_Date', 'Progress_Of_Implementation', 'Social_Status']

#### 3.1 Combining all available TS-iPASS Data

In [24]:
ipass_DF = None

for file in tqdm(glob('Data/raw/TS_iPASS_Data/*')):
    try:
        temp = pd.read_csv(file).rename(columns = ipass_column_names)
    except:
        print(f"Unabale to read the file -> {file}")
    finally:
        temp.application_date = pd.to_datetime(temp.application_date, format = '%d/%m/%Y')
        temp.approval_date = pd.to_datetime(temp.approval_date, format = '%d/%m/%Y')
        if not isinstance(ipass_DF, pd.DataFrame):
            ipass_DF = temp.copy(deep = True)
        else:
            ipass_DF = pd.concat([ipass_DF, temp])

# Renaming columns
ipass_DF.columns = ipass_DF.columns.str.title()

# Selecting needed columns only
ipass_DF = ipass_DF[registration_selected_columns]

# Unifying Values
ipass_DF.District_Name = ipass_DF.District_Name.replace({value:
                            key for key, values in district_mapping.items() for value in values})
ipass_DF.Name_Of_The_Unit = ipass_DF.Name_Of_The_Unit.str.title()
ipass_DF.Progress_Of_Implementation = ipass_DF.Progress_Of_Implementation.str.title()

print(f'Number of Districts in TS-iPASS Dataset\t\t: {ipass_DF.District_Name.nunique()}')
print(f'Number of data points in the  TS-iPASS Dataset\t: {ipass_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t: {sum(ipass_DF.isnull().sum())}\n')

ipass_DF = ipass_DF.sort_values('Application_Date').reset_index(drop = True)
ipass_DF.head()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 182.98it/s]


Number of Districts in TS-iPASS Dataset		: 33
Number of data points in the  TS-iPASS Dataset	: 23267
Number of missing values in the dataset		: 3



,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
0,Ranga Reddy,Jagriti Steel Private Limited,Steel and steel products using various furnace...,Engineering,5.55,45,2016-04-10,2017-08-22,Commenced Operations,General
1,Medchal-Malkajgiri,R K Distilleries Private Limited,Distillery ( molasses / grain /yeast based),Beverages,10.00,460,2016-04-10,2017-04-07,Commenced Operations,SC
2,Siddipet,Svr Seating Systems,Steel furniture without spray painting,Engineering,4.04,28,2016-04-13,2017-08-22,Commenced Operations,General
3,Medak,Mahalakshmi Profiles Private Ltd,Electrical and electronic item assembling ( co...,Electrical and Electronic Products,6.00,52,2016-04-26,2017-08-01,Commenced Operations,ST
4,Suryapet,Ncl Industries Ltd,Cement,"Cement, Cement & Concrete Products, Fly Ash Br...",50.00,200,2016-04-27,2017-10-31,Commenced Operations,ST


#### 3.2 Handling Date-Time data and defining time period

- Start point : `01-01-2019`
- End point   : `31-12-2024`

In [25]:
ipass_DF = ipass_DF.query('2019 <= Application_Date < 2025')

# Check for applications where the Approval_Date is earlier than the Application_Date
display(ipass_DF.query('Application_Date > Approval_Date'))

# Removing all applications where the Approval_Date is earlier than the Application_Date
ipass_DF = ipass_DF.query('Application_Date <= Approval_Date')
ipass_DF = ipass_DF.sort_values(['Application_Date', 'District_Name']).reset_index(drop = True)

print(f'\n>>> Number of datapoints in the revised dataset\t\t: {ipass_DF.shape[0]}')
print(f'>>> Number of missing values in the revised dataset\t: {sum(ipass_DF.isnull().sum())}\n')
ipass_DF.head()

,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
5332,Medak,Raj Udyog,Industry or process involving'metal surface tr...,Others,1.4627,15,2019-01-11,2018-06-20,Commenced Operations,General
7491,Medchal-Malkajgiri,Ferring Laboratories Private Limited,Pharmaceutical Formulation for R & D purpose (...,R&D,93.0000,50,2019-10-09,2019-04-17,Initial Stage,General



>>> Number of datapoints in the revised dataset		: 18000
>>> Number of missing values in the revised dataset	: 3



,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
0,Medchal-Malkajgiri,Navateja Marketing Pvt Ltd,Manufacturing of iodized salt fromcrude/ raw salt,Food Processing,0.1200,10,2019-01-01,2019-07-19,Commenced Operations,General
1,Ranga Reddy,Sri Koteswara Cam Systems Pvt Ltd,Engineering and fabrication units (dry process...,Engineering,9.1100,125,2019-01-01,2019-04-17,Commenced Operations,General
2,Sangareddy,Sanjay Technical Services Private Ltd,Engineering and fabrication units (dry process...,Engineering,5.0550,40,2019-01-01,2019-05-02,Advanced Stage,General
3,Jagtial,Nishan Singh Engineering Works,Engineering and fabrication units (dry process...,Engineering,0.0000,2,2019-01-02,2019-01-16,Commenced Operations,OBC
4,Jangaon,Dew Industries,Engineering and fabrication units (dry process...,Engineering,0.4916,8,2019-01-02,2019-01-11,Commenced Operations,General


#### 3.3 Handling Missing Values

In [26]:
print(f'Number of missing values in the dataset : {sum(ipass_DF.isnull().sum())}\n')
ipass_DF.isnull().sum()

Number of missing values in the dataset : 3



District_Name                 0
Name_Of_The_Unit              0
Line_Of_Activity              0
Sector                        3
Investment                    0
Number_Of_Employees           0
Application_Date              0
Approval_Date                 0
Progress_Of_Implementation    0
Social_Status                 0
dtype: int64

In [27]:
missingDF = ipass_DF.query('Sector != Sector')
missingDF.style.map(style_nan, props = 'color:red;', subset = ['Sector'])

,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
6662,Medchal-Malkajgiri,G.V. Research Centers Private Limited,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes",nan,0.500000,10,2021-02-16 00:00:00,2021-12-17 00:00:00,Yet To Start Construction,General
10189,Medchal-Malkajgiri,Crescentia Labs Private Limited,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes",nan,6.390000,20,2022-01-01 00:00:00,2022-01-20 00:00:00,Yet To Start Construction,General
11780,Karimnagar,M/S. Sri Venkateshwara Godowns,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes",nan,0.250000,12,2022-05-30 00:00:00,2022-06-24 00:00:00,Yet To Start Construction,OBC


Filtering the corresponding `Line_Of_Activity` to see the `Sector` values of other industries.

In [28]:
Line_Of_Activity = missingDF.Line_Of_Activity.values[0]
ipass_DF.query('Line_Of_Activity == @Line_Of_Activity').style.map(style_nan,
                                                        props = 'color:red;', subset = ['Sector'])

,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
6662,Medchal-Malkajgiri,G.V. Research Centers Private Limited,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes",nan,0.500000,10,2021-02-16 00:00:00,2021-12-17 00:00:00,Yet To Start Construction,General
8136,Karimnagar,M/S. Primary Agriculture Co-Operative Society Ltd.,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes","Real Estate,Industrial Parks and IT Buildings",0.225900,10,2021-06-25 00:00:00,2022-03-08 00:00:00,Yet To Start Construction,General
10189,Medchal-Malkajgiri,Crescentia Labs Private Limited,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes",nan,6.390000,20,2022-01-01 00:00:00,2022-01-20 00:00:00,Yet To Start Construction,General
11780,Karimnagar,M/S. Sri Venkateshwara Godowns,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes",nan,0.250000,12,2022-05-30 00:00:00,2022-06-24 00:00:00,Yet To Start Construction,OBC
12012,Karimnagar,M/S. Pasha Mango Fruit Company Godown,"Industrial estates/ parks/ complexes/ areas, Export Processing Zones (EPZs), Special Economic Zones (SEZs), Biotech parks, Leather complexes","Real Estate,Industrial Parks and IT Buildings",0.226300,8,2022-06-21 00:00:00,2022-07-04 00:00:00,Yet To Start Construction,OBC


In [29]:
ipass_DF.loc[ipass_DF.Name_Of_The_Unit == 'Crescentia Labs Private Limited', 'Sector'] = \
                                                    'Real Estate,Industrial Parks and IT Buildings'
ipass_DF.loc[ipass_DF.Name_Of_The_Unit == 'M/S. Sri Venkateshwara Godowns', 'Sector'] = \
                                                    'Real Estate,Industrial Parks and IT Buildings'
ipass_DF.loc[ipass_DF.Name_Of_The_Unit == 'G.V. Research Centers Private Limited', 'Sector'] = \
                                                    'Real Estate,Industrial Parks and IT Buildings'

print(f'Number of missing values in the revised dataset : {sum(ipass_DF.isnull().sum())}')

Number of missing values in the revised dataset : 0


#### 3.4 Type-Casting to correct datatype

In [30]:
# Type-Casting to correct datatype

ipass_DF.District_Name = ipass_DF.District_Name.astype('string')
ipass_DF.Name_Of_The_Unit = ipass_DF.Name_Of_The_Unit.astype('string')
ipass_DF.Line_Of_Activity = ipass_DF.Line_Of_Activity.astype('string')
ipass_DF.Sector = ipass_DF.Sector.astype('string')
ipass_DF.Progress_Of_Implementation = ipass_DF.Progress_Of_Implementation.astype('string')
ipass_DF.Social_Status = ipass_DF.Social_Status.astype('string')

display(ipass_DF.info())
ipass_DF.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   District_Name               18000 non-null  string        
 1   Name_Of_The_Unit            18000 non-null  string        
 2   Line_Of_Activity            18000 non-null  string        
 3   Sector                      18000 non-null  string        
 4   Investment                  18000 non-null  float64       
 5   Number_Of_Employees         18000 non-null  int64         
 6   Application_Date            18000 non-null  datetime64[ns]
 7   Approval_Date               18000 non-null  datetime64[ns]
 8   Progress_Of_Implementation  18000 non-null  string        
 9   Social_Status               18000 non-null  string        
dtypes: datetime64[ns](2), float64(1), int64(1), string(6)
memory usage: 1.4 MB


None

,District_Name,Name_Of_The_Unit,Line_Of_Activity,Sector,Investment,Number_Of_Employees,Application_Date,Approval_Date,Progress_Of_Implementation,Social_Status
0,Medchal-Malkajgiri,Navateja Marketing Pvt Ltd,Manufacturing of iodized salt fromcrude/ raw salt,Food Processing,0.1200,10,2019-01-01,2019-07-19,Commenced Operations,General
1,Ranga Reddy,Sri Koteswara Cam Systems Pvt Ltd,Engineering and fabrication units (dry process...,Engineering,9.1100,125,2019-01-01,2019-04-17,Commenced Operations,General
2,Sangareddy,Sanjay Technical Services Private Ltd,Engineering and fabrication units (dry process...,Engineering,5.0550,40,2019-01-01,2019-05-02,Advanced Stage,General
3,Jagtial,Nishan Singh Engineering Works,Engineering and fabrication units (dry process...,Engineering,0.0000,2,2019-01-02,2019-01-16,Commenced Operations,OBC
4,Jangaon,Dew Industries,Engineering and fabrication units (dry process...,Engineering,0.4916,8,2019-01-02,2019-01-11,Commenced Operations,General


#### 3.5 Saving TS-iPASS Dataset

In [31]:
# Saving DataFrame to Parquet

TSiPASS_path = f'{interim_file_path}/TSiPASS_Data.parquet'
print(f'Saving cleaned TS-iPASS Dataset to "{TSiPASS_path}"')
if not os.path.isfile(TSiPASS_path):
    ipass_DF.to_parquet(TSiPASS_path, engine = 'pyarrow')

Saving cleaned TS-iPASS Dataset to "Data/interim/TSiPASS_Data.parquet"


### 4. RTA Vehicle Online Sales Data

**Column Details**

- `District` : Vehicle Registration District name
- `Model_Desc` : Model Description of the vehicle
- `vehicleClass` : Vehicle Classification
- `Fuel` : Fuel type
- `fromdate` : Date of registration of the vehicle
- `TempRegnNo` : Temporary Registration Number
- `Category` : If the vehicle is Transport or Non Transport
- `SecondVehicle` : Is it a second vehicle of the owner
- `Make_Yr` : Vehicle Make Year
- `Manufacturer_Name`: Name of the Automobile Company

In [32]:
# https://www.transport.telangana.gov.in/html/aboutus-contactdirectory-district-offices.html
# https://paytminsurance.co.in/rto/telangana/siddipet-ts-36/
# https://www.policybazaar.com/rto/telangana/
# https://groww.in/rto/telangana

# https://transport.telangana.gov.in/html/reservationnumber.php : for numbers starts with TGxxx
rto_mapping = {'Adilabad' : ['RTA ADILABAD', 'TS001'],
               'Bhadradri Kothagudem' : ['RTA BHADRADRI', 'UNIT OFFICE BHADRACHALAM',
                                         'TS028', 'TS128'],
               'Hanumakonda' : ['RTA HANUMAKONDA'],
               'Hyderabad' : ['RTA-HYDERABAD-CZ', 'RTA-HYDERABAD-WZ', 'RTA-HYDERABAD-NZ',
                              'RTA-HYDERABAD-SZ', 'RTA-HYDERABAD-EZ', 'TS009', 'TS010', 'TS011',
                              'REGIONAL TRANSPORT OFFICER, HYDERABAD', 'TS012', 'TS013'],
               'Jagtial' : ['RTA JAGITYAL', 'UNIT OFFICE KORUTLA', 'TS021', 'TS121'],
               'Jangaon' : ['RTA JANGOAN', 'TS027'],
               'Jayashankar Bhupalpally' : ['RTA JAYASHANKAR', 'TS025'],
               'Jogulamba Gadwal' : ['RTA JOGULAMBA', 'TS033'],
               'Kamareddy' : ['RTA KAMAREDDY', 'TS017'],
               'Karimnagar' : ['RTA KARIMNAGAR', 'UNIT OFFICE HUZURABAD', 'TS002', 'TS102'],
               'Khammam' : ['RTA KHAMMAM', 'UNIT OFFICE SATTUPALLI', 'UNIT OFFICE WYRA',
                            'TS004', 'TS304', 'TS404'],
               'Komaram Bheem Asifabad' : ['RTA KOMRAMBHEEM', 'TS020'],
               'Mahabubabad' : ['RTA MAHABUBABAD', 'TS026'],
               'Mahabubnagar' : ['RTA MAHABOOBNAGAR', 'TS006'],
               'Mancherial' : ['RTA MANCHERIAL', 'TS019'],
               'Medak' : ['RTA MEDAK', 'TS035'],
               'Medchal-Malkajgiri' : ['RTA MEDCHAL', 'RTA UPPAL', 'UNIT OFFICE KUKATPALLY',
                                       'TS008', 'TS108', 'TS208'],
               'Mulugu' : ['RTA MULUG', 'RTA MULUGU'],
               'Nagarkurnool' : ['RTA NAGARKURNOOL', 'UNIT OFFICE KALWAKURTHY', 'TS031', 'TS131'],
               'Nalgonda' : ['RTA NALGONDA', 'UNIT OFFICE MIRYALAGUDA', 'TS005', 'TS305'],
               'Narayanpet' : ['RTA NARAYANPET', 'TS038'],
               'Nirmal' : ['RTA NIRMAL', 'TS018'],
               'Nizamabad' : ['RTA NIZAMABAD', 'UNIT OFFICE ARMOOR', 'UNIT OFFICE BHODAN',
                              'TS016', 'TS116', 'TS216'],
               'Peddapalli' : ['RTA PEDDAPALLI', 'UNIT OFFICE RAMAGUNDAM', 'TS022', 'TS122'],
               'Rajanna Sircilla' : ['RTA RAJANNA', 'TS023'],
               'Ranga Reddy' : ['RTA RANGAREDDY', 'RTA IBRAHIMPATNAM', 'UNIT OFFICE SHADNAGAR',
                                'TS007', 'TS107', 'TS207'],
               'Sangareddy' : ['UNIT OFFICE PATANCHERUVU', 'RTA SANGAREDDY',
                               'UNIT OFFICE ZAHIRABAD', 'TS015', 'TS215', 'TS415'],
               'Siddipet' : ['RTA SIDDIPET', 'TS036'],
               'Suryapet' : ['RTA SURYAPET', 'UNIT OFFICE KODAD', 'TS029', 'TS129'],
               'Vikarabad' : ['RTA VIKARABAD', 'UNIT OFFICE PARGI', 'TS034', 'TS134'],
               'Wanaparthy' : ['RTA WANAPARTHY', 'UNIT OFFICE PEBBAIR', 'TS032', 'TS132'],
               'Warangal' : ['RTA WARANGAL RURAL', 'RTA WARANGAL URBAN', 'RTA WARANGAL',
                             'TS003', 'TS024'],
               'Yadadri Bhuvanagiri' : ['RTA YADADRI', 'TS030']}

rta_mapping = {'makeYear' : ['Make_Yr', 'makeyear'],
               'SeatingCapacity' : ['SeatingCapacity', 'seatCapacity'],
               'InsuranceValidity' : ['IncValidTo', 'insuranceValidity'],
               'vehicleClass' : ['V_Vhc_ClsID', 'ClassofVeh'],
               'Model_Desc' : ['modelDesc'],
               'Fuel' : ['fuel'],
               'Colour' : ['colour'],
               'SecondVehicle' : ['secondVehicle'],
               'TempRegnNo' : ['tempRegistrationNumber'],
               'Category' : ['category', 'C_Transport'],
               'Manufacturer_Name' : ['makerName'],
               'slno' : ['rowid'],
               'fromdate' : ['Apprved_Dt'],
               'OfficeCd' : ['OfficeCd']}

rta_selected_columns = ['District', 'Model_Desc', 'vehicleClass', 'Fuel', 'SeatingCapacity',
                        'fromdate', 'TempRegnNo', 'Category', 'SecondVehicle',
                        'makeYear', 'Manufacturer_Name'] # Colour

#### 4.1 Combining all available RTA Data

In [33]:
rta_DF = None

for file in tqdm(glob('Data/raw/RTA_Online_Sales/*')):
    try:
        temp = pd.read_csv(file).rename(columns = {value:
                            key for key, values in rta_mapping.items() for value in values})
    except:
        temp = pd.read_csv(file, encoding = 'ISO-8859-1').rename(columns = {value:
                            key for key, values in rta_mapping.items() for value in values})
    finally:
        # try: temp = temp.query('makeYear != "00:00.0"').copy(deep = True)
        # except: pass
        if 'OfficeCd.1' in temp.columns:
            temp.drop('OfficeCd', axis = 1, inplace = True)
            temp.rename(columns = {'OfficeCd.1' : 'OfficeCd'}, inplace = True)
        if 'fromdate' not in temp.columns:
            temp['fromdate'] = file.strip('.csv').split('to')[-1].replace('_', '/')
        if re.search(r'\d{2}\/\d{2}\/\d{4}', temp.fromdate.values[0]): # 01/06/2021
            try: temp.fromdate =  pd.to_datetime(temp.fromdate, format = '%d/%m/%Y')
            except: temp.fromdate =  pd.to_datetime(temp.fromdate, format = 'mixed')
        elif re.search(r'\d{2}\/\d{2}\/\d{2}', temp.fromdate.values[0]): # 01/01/18
            temp.fromdate = pd.to_datetime(temp.fromdate, format = '%d/%m/%y')
        if not isinstance(rta_DF, pd.DataFrame):
            rta_DF = temp.copy(deep = True)
        else:
            rta_DF = pd.concat([rta_DF, temp], join = 'outer')

print(f'Number of data points in the RTA Registration Dataset\t: {rta_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t\t: {sum(rta_DF.isnull().sum())}\n')

rta_DF.head()

/tmp/ipykernel_410187/1637960469.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(file).rename(columns = {value:
100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 74/74 [01:02<00:00,  1.18it/s]


Number of data points in the RTA Registration Dataset	: 9748539
Number of missing values in the dataset			: 29368411



,slno,Model_Desc,Fuel,Colour,vehicleClass,makeYear,SeatingCapacity,InsuranceValidity,SecondVehicle,TempRegnNo,Category,Manufacturer_Name,OfficeCd,fromdate,todate,SLNO,DealerName,InsuranceCompany
0,8550857.0,RE COMPACT LPG THL BSIV,PETROL LPG,GOLDEN YELLOW,Auto Rickshaw,01/11/2019,4.0,30/03/2021,N,TS09CYTR5999,Transport,BAJAJ AUTO LTD,RTA YADADRI,2020-04-01,02/04/2020,NaN,NaN,NaN
1,989436.0,JOHN DEERE 5050D V3 TRACTOR BSIIIA,DIESEL,GREEN,Tractor for Agricultural Purpose,01/03/2020,1.0,01/04/2021,N,TS16AFTR8888,Non Transport,JOHN DEERE INDIA PVT LIMITED,UNIT OFFICE BHODAN,2020-04-01,02/04/2020,NaN,NaN,NaN
2,5376491.0,SWARAJ 742 FE TRACTOR BSIIIA,DIESEL,BLUE,Tractor for Agricultural Purpose,01/03/2020,1.0,30/03/2021,N,TS05AMTR5297,Non Transport,MAHINDRA & MAHINDRA LIMITED,RTA NALGONDA,2020-04-01,02/04/2020,NaN,NaN,NaN
3,1449153.0,SWARAJ 735 FE,DIESEL,BLUE,Tractor for Agricultural Purpose,01/03/2020,1.0,30/03/2021,N,TS05AMTR5296,Non Transport,MAHINDRA & MAHINDRA LIMITED,RTA NALGONDA,2020-04-01,02/04/2020,NaN,NaN,NaN
4,2781713.0,SWARAJ 744 FE,DIESEL,BLUE,Tractor for Commercial Use,01/03/2020,1.0,30/03/2021,N,TS05AMTR5295,Transport,MAHINDRA & MAHINDRA LIMITED,RTA NALGONDA,2020-04-01,02/04/2020,NaN,NaN,NaN


#### 4.2 Unifying Fuel Values

In [34]:
def unify_fuel(row: str) -> str:
    if not isinstance(row, str):
        return row
    row = re.sub(r'PETROL LPG|PETROL ELECTRIC|DIESEL ELECTRIC', 'Hybrid', row)
    row = re.sub(r'CNG PETROL|DIESEL LPG|LPG PETROL|PETROL CNG', 'Hybrid', row)
    row = re.sub(r'BATTERY', 'Electric', row)
    row = re.sub(r'LPG|CNG', 'LPG/CNG', row)
    if '/' not in row:
        return row.title()
    return row

In [35]:
rta_DF.Fuel = rta_DF.Fuel.progress_apply(unify_fuel)
rta_DF.vehicleClass = rta_DF.vehicleClass.str.title()

# Unifying values in 'OfficeCd' and correcting column name to 'District'
rta_DF.OfficeCd = rta_DF.OfficeCd.replace(
                        {value: key for key, values in rto_mapping.items() for value in values})
rta_DF.rename(columns = {'OfficeCd' : 'District'}, inplace = True)

# Removing leading and trailing whitespaces
rta_DF.loc[:, 'Model_Desc'] = rta_DF['Model_Desc'].str.strip()

rta_DF.head()

100%|█████████████████████████████████████████████████████████████████████████████████████| 9748539/9748539 [00:25<00:00, 389502.23it/s]


,slno,Model_Desc,Fuel,Colour,vehicleClass,makeYear,SeatingCapacity,InsuranceValidity,SecondVehicle,TempRegnNo,Category,Manufacturer_Name,District,fromdate,todate,SLNO,DealerName,InsuranceCompany
0,8550857.0,RE COMPACT LPG THL BSIV,Hybrid,GOLDEN YELLOW,Auto Rickshaw,01/11/2019,4.0,30/03/2021,N,TS09CYTR5999,Transport,BAJAJ AUTO LTD,Yadadri Bhuvanagiri,2020-04-01,02/04/2020,NaN,NaN,NaN
1,989436.0,JOHN DEERE 5050D V3 TRACTOR BSIIIA,Diesel,GREEN,Tractor For Agricultural Purpose,01/03/2020,1.0,01/04/2021,N,TS16AFTR8888,Non Transport,JOHN DEERE INDIA PVT LIMITED,Nizamabad,2020-04-01,02/04/2020,NaN,NaN,NaN
2,5376491.0,SWARAJ 742 FE TRACTOR BSIIIA,Diesel,BLUE,Tractor For Agricultural Purpose,01/03/2020,1.0,30/03/2021,N,TS05AMTR5297,Non Transport,MAHINDRA & MAHINDRA LIMITED,Nalgonda,2020-04-01,02/04/2020,NaN,NaN,NaN
3,1449153.0,SWARAJ 735 FE,Diesel,BLUE,Tractor For Agricultural Purpose,01/03/2020,1.0,30/03/2021,N,TS05AMTR5296,Non Transport,MAHINDRA & MAHINDRA LIMITED,Nalgonda,2020-04-01,02/04/2020,NaN,NaN,NaN
4,2781713.0,SWARAJ 744 FE,Diesel,BLUE,Tractor For Commercial Use,01/03/2020,1.0,30/03/2021,N,TS05AMTR5295,Transport,MAHINDRA & MAHINDRA LIMITED,Nalgonda,2020-04-01,02/04/2020,NaN,NaN,NaN


#### 4.3 Handling Date-Time data and defining time period

- Start point : `01-01-2019`
- End point   : `31-12-2024`

In [36]:
# Type-Casting to correct datatypes
rta_DF.fromdate = pd.to_datetime(rta_DF.fromdate)
rta_DF.makeYear = pd.to_datetime(rta_DF.makeYear, format = 'mixed')

# Defining date range of the data
rta_DF = rta_DF.query('fromdate == fromdate and 2019 <= fromdate < 2025')
rta_DF = rta_DF[rta_selected_columns].sort_values('fromdate').reset_index(drop = True)

print(f'Number of Districts in RTA Registration Dataset\t\t: {rta_DF.District.nunique()}')
print(f'Number of data points in the RTA Registration Dataset\t: {rta_DF.shape[0]}')
print(f'Number of missing values in the dataset\t\t\t: {sum(rta_DF.isnull().sum())}\n')

rta_DF.head()

Number of Districts in RTA Registration Dataset		: 33
Number of data points in the RTA Registration Dataset	: 9608324
Number of missing values in the dataset			: 218206



,District,Model_Desc,vehicleClass,Fuel,SeatingCapacity,fromdate,TempRegnNo,Category,SecondVehicle,makeYear,Manufacturer_Name
0,Warangal,CLASSIC 350 ABS BSIV,Motor Cycle,Petrol,2.0,2019-01-01,TS05ADTR8900,Non Transport,N,2018-01-11,ROYAL ENFIELD
1,Hyderabad,TATA NEXON-XZA 1.5 RTQ BSIV,Motor Car,Diesel,5.0,2019-01-01,TS09CCTR0495,Non Transport,N,2018-01-07,TATA MOTORS LTD
2,Sangareddy,TATA TIGOR XZ+ 1.2 RTN BSIV BSIV,Motor Car,Petrol,5.0,2019-01-01,TS09CCTR0092,Non Transport,Y,2018-01-10,TATA MOTORS LTD
3,Sangareddy,TVS NTORQ 125 DISC BSIV,Motor Cycle,Petrol,2.0,2019-01-01,TS15VTR0854,Non Transport,N,2018-01-08,TVS MOTOR COMPANY LTD
4,Ranga Reddy,GRAZIA WEAS&KS&FDISC/REAR DCBS(CBS)WALLOYW BSIV,Motor Cycle,Petrol,2.0,2019-01-01,TS08AZTR2977,Non Transport,N,2018-01-02,HONDA MOTORCYCLE&SCOOTER(I)P L


#### 4.4 Handling Missing Values

In [37]:
print(f'Number of missing values in the dataset : {sum(rta_DF.isnull().sum())}')
rta_DF.isnull().sum()

Number of missing values in the dataset : 218206


District                  2
Model_Desc                2
vehicleClass              2
Fuel                 218188
SeatingCapacity           2
fromdate                  0
TempRegnNo                2
Category                  2
SecondVehicle             2
makeYear                  2
Manufacturer_Name         2
dtype: int64

In [38]:
print('>>> Rows where District name is not available\n')
display(rta_DF.query('District != District'))

# Removing rows where District is not available
rta_DF = rta_DF.query('District == District')
rta_DF = rta_DF.sort_values(['fromdate', 'District']).reset_index(drop = True)

print(f'\n>>> Number of data points in the RTA Registration Dataset : {rta_DF.shape[0]}')
print(f'>>> Number of missing values in the revised dataset\t  : {sum(rta_DF.isnull().sum())}')

>>> Rows where District name is not available



,District,Model_Desc,vehicleClass,Fuel,SeatingCapacity,fromdate,TempRegnNo,Category,SecondVehicle,makeYear,Manufacturer_Name
6860998,NaN,NaN,NaN,NaN,NaN,2022-10-31,NaN,NaN,NaN,NaT,NaN
6860999,NaN,NaN,NaN,NaN,NaN,2022-10-31,NaN,NaN,NaN,NaT,NaN



>>> Number of data points in the RTA Registration Dataset : 9608322
>>> Number of missing values in the revised dataset	  : 218186


In [39]:
missingDF = rta_DF.query('Fuel != Fuel').reset_index(drop = True)
print(f'Number of rows where the Fuel cell is missing : {missingDF.shape[0]}\n')
missingDF.sample(5, random_state = 1).style.map(style_nan, props = 'color:red;',
                                                 subset = ['Fuel'])

Number of rows where the Fuel cell is missing : 218186



,District,Model_Desc,vehicleClass,Fuel,SeatingCapacity,fromdate,TempRegnNo,Category,SecondVehicle,makeYear,Manufacturer_Name
19102,Suryapet,5TON 2WHEEL SEMI TIPPING TRAILER IRON BODY,Trailer For Agriculture Purpose,nan,0.000000,2019-07-11 00:00:00,TS26TR4539,Non Transport,N,2019-01-06 00:00:00,M/S AMEER ENGINEERING WORKS MAHABUBABAD
124024,Suryapet,5TONS 2WHEELER SEMI TIIPING TRAILER,Trailer For Commercial Use,nan,0.000000,2021-06-14 00:00:00,TS29GTR8341,Transport,N,2021-01-05 00:00:00,M/S SRI PARAMESHWARA INDUSTRIES
104424,Mahabubnagar,5 TONNS 2WH SEMI TRAILER TIPPING IRON BODY,Trailer For Commercial Use,nan,0.000000,2021-02-05 00:00:00,TS32CTR0879,Transport,N,2020-01-10 00:00:00,SAAMIRI ENGG WORKS MBNAGAR
48011,Wanaparthy,5TONS 2WHEELER SEMI TIPPING TRAILER,Trailer For Agriculture Purpose,nan,0.000000,2020-03-15 00:00:00,TS32BTR5117,Non Transport,N,2019-01-04 00:00:00,"M/S NVR ENGINEERING WORKS, WANAPARTHY"
54037,Nagarkurnool,TWO WHEELER TRAILOR TIPPING - 5 TONNS,Trailer For Commercial Use,nan,0.000000,2020-05-29 00:00:00,TS06AFTR8199,Transport,N,2020-01-04 00:00:00,M/S SRI BADESHWARA ARGO INDUSTRIES


In [40]:
missingDF = pd.DataFrame(missingDF.Model_Desc.value_counts()).reset_index()
missingDF.rename(columns ={'count' : 'Missing_Count'}, inplace = True)
print('>>> Top 10 vehicles, where Fuel entry is missing.\n') 
missingDF.head(10)

>>> Top 10 vehicles, where Fuel entry is missing.



,Model_Desc,Missing_Count
0,5TONS 2WHEELER SEMI TIPPING TRAILER,29547
1,TWO WHEELER TRAILOR TIPPING - 5 TONNS,21636
2,5 TONS 2 WHEELER SEMI TIPPING T IRON BODY,20337
3,TWO WHEELER SEMI TRAILOR TIPPING - 5 TONNS,8848
4,5 TONS 2 WHEELER SEMI TIPPING IRON BODY,7511
5,5TONNS 2WH SEMI TRAILER TIPPING IRON BODY,7040
6,5 TONS 2WH SEMI TRAILER TIPPING IRON BODY,6294
7,5 TONS 2 WHEELER WATER TANKER T IRON BODY,6041
8,TACTOR 2WHEELER SEMITRAILOR TIPPING - 5 TONNS,6035
9,5 TONS 2WHEEL SEMI TIPPING TRAILER,5774


**Filling missing values using vehicle model name**

In [41]:
# https://www.bikewale.com/hero-bikes/maestro/
rta_DF.loc[rta_DF.Model_Desc == 'MAESTRO EDGE 125(DRUM-CAST) BSIV', 'Fuel'] = 'Petrol'
# https://www.bikewale.com/hero-bikes/maestro/
rta_DF.loc[rta_DF.Model_Desc == 'MAESTRO EDGE 125 (DISC-CAST) BSIV', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/alto-800/vxi/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI- ALTO VXI BSVI', 'Fuel'] = 'Petrol'
# https://www.carwale.com/land-rover-cars/defender/
rta_DF.loc[rta_DF.Model_Desc == 'DEFENDER 3.0 DIESEL 4WD AUTO 5DOOR SE 5STR', 'Fuel'] = 'Diesel'
# https://www.carwale.com/mahindra-cars/bolero-2011-2020/zlx-bs-iv/
rta_DF.loc[rta_DF.Model_Desc == 'MBOLEROLX 4WD 7STR OPTACOPT PS ABS BSIV', 'Fuel'] = 'Diesel'
# https://www.carwale.com/mahindra-cars/tuv300/t10/
rta_DF.loc[rta_DF.Model_Desc == 'MAHINDRA TUV300 T10 MHAWK100 BSIV', 'Fuel'] = 'Diesel'
# https://www.carwale.com/maruti-suzuki-cars/dzire-2017-2020/vdi-amt/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI -VITARA BREZZA VDI AMT BSIV', 'Fuel'] = 'Diesel'
# https://www.carwale.com/tata-cars/harrier-2019-2023/xza-plus/
rta_DF.loc[rta_DF.Model_Desc == 'HARRIER XZA- +2LBS6 BSVI-PH2', 'Fuel'] = 'Diesel'
# https://www.carwale.com/tata-cars/indigo-ecs-2013-2018/gls/
rta_DF.loc[rta_DF.Model_Desc == 'INDIGO ECS GLS MPFI BSIV', 'Fuel'] = 'Petrol'
# https://www.carwale.com/land-rover-cars/discovery/20-petrol/
rta_DF.loc[rta_DF.Model_Desc == 'DISCOVERY 2.0 PETROL 4WD AUTO 5DOOR SE7STR BSIV', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/alto-800/lxi-o/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI- ALTO LXI (O) BSVI', 'Fuel'] = 'Petrol'
# https://www.carwale.com/hyundai-cars/tucson/
rta_DF.loc[rta_DF.Model_Desc == 'TUCSON CRDI AUTO GLS BSVI', 'Fuel'] = 'Diesel'
# https://tractorkarvan.com/tractor/powertrac-euro-55-next-4wd
rta_DF.loc[rta_DF.Model_Desc == 'EURO 55 E19 (BRAND NAME-POWERTRAC) 4WD BSIIIA', 'Fuel'] = 'Diesel'
# https://tractorkarvan.com/tractor/force-sanman-6000
rta_DF.loc[rta_DF.Model_Desc == 'SANMAN 6000 BS IIIA', 'Fuel'] = 'Diesel'
# Maruti Ciaz is only mild-hybrid, not fully hybrid
# https://www.carwale.com/maruti-suzuki-cars/ciaz/alpha-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI CIAZ SMART HYBIRD AUTOMATIC ALPHA BSIV', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/ciaz/delta-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI - CIAZ SMART HYBRID DELTA 1.5L 5MT BSVI-PH2', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/ciaz/alpha-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI - CIAZ SMART HYBRID ALPHA  1.5L AT BSVI-PH2', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/ciaz/sigma-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI - CIAZ SMART HYBRID SIGMA 1.5L 5MT BSVI', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/ciaz/zeta-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI - CIAZ SMART HYBRID ZETA  1.5L AT BSVI-PH2', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/ciaz/zeta-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI - CIAZ SMART HYBRID ZETA 1.5L 5MT BSVI-PH2', 'Fuel'] = 'Petrol'
# https://www.carwale.com/maruti-suzuki-cars/ciaz/alpha-15/
rta_DF.loc[rta_DF.Model_Desc == 'MARUTI - CIAZ SMART HYBRID ALPHA 1.5L 5MT BSVI-PH2', 'Fuel'] = 'Petrol'
# https://trucks.cardekho.com/en/trucks/mahindra/alfa/3-seatercomfy
rta_DF.loc[rta_DF.Model_Desc == 'MAHINDRA ALFA PAX CAMPY DIESEL BSIV', 'Fuel'] = 'Diesel'

rta_DF.loc[rta_DF.Model_Desc == 'DASMESH-912 COMBINE HARVESTER', 'Fuel'] = 'Diesel'
rta_DF.loc[rta_DF.Model_Desc == 'DASMESH-912COMIBNED HARVESTER', 'Fuel'] = 'Diesel'

tractor_trailer = ['5TON 2WH TRAILERS',
                   '5TON 2WH SEMI TRAILER TIPPING',
                   'TWO WHEELER  TRACTOR TRAILOR TIPPING - 5 TONNS',
                   'TRACTOR TRAILER 5 TONNS 2WH',
                   'TWO WHEELER SEMI TRAILOR TIPPING - 5 TONNS',
                   'TRACTORTRAILERSEMI  5TONS (2WHEELER)',
                   '5 TONS OF 2 WHEELER TIPPING SEMI TRAILER',
                   '3TONS2WHELLERSEMITIPPING TRAILER IB',
                   '5 TONS TWO WHEELER TIPPING TRAILER',
                   '5TON 2WH TIPPING TYPE TRAIALERS',
                   '5TONNS 2WH TIPPING TRAILER',
                   '5 TONS 2 WHEELER TIPPING TRAILERIRONBODY',
                   '5 TON 2 WH SEMI TRAILER TIPPING IRON BODY BSIV',
                   '5 TONS 2WHEELER TIPPING SEMI TRAILER',
                   '5TON 2 WHEELR SEMI TIPPING TRAILER',
                   '3TONS 2WHEELER SEMITIPPING TRAILER',
                   '5 TON 2WH TIPPING TRAILER',
                   '5 TONS 2 WHEELER SEMI TIPPING TRAILER IRON BODY',
                   '5 TONS OF TWO WHEELERTIPPING SEMI TRAILER',
                   '5 TONNS 2WH SEMI TRAILER TIPPING IRON BODY',
                   '5TONS 2WHEELER SEMI TIIPING TRAILER',
                   '5 TONS 2WHEELER SEMI TIPPING TRAILER',
                   'WATER TANKER  5TONS  2WHEELER TRAILER',
                   '5TONS 2WHEELER TIPPING TRAILER IRON BODY',
                   'TWO WHEELED TRAILER',
                   '5TON 2WH SEMI TR TIPPING IRON BODY',
                   '5TONS 2WHEER SEMI TIPPING TRAILER',
                   '5 TONS 2 WHEELER TIPPING TRAILER',
                   '5 TONS 2 WHEELER SEMI TIPPING T IRON BODY',
                   '5 TON 2 WHEELER  TIPPING TRAILER',
                   '5 TONS OF 2 WHEELER SEMI-TIPPING TRAILOR',
                   '5 TONN 2WH TRAILER',
                   '5TONS OF 2WHEELER SEMI TIPPING TRAILER',
                   '5TON 2WH TRAILER SEMI TR TIPPING IRON BODY',
                   '5 TON 2WH SEMI TRAILER TIPPING IRON BODY',
                   '5 TONS 2WH SEMI TRAILER TIPPING IRON BODY',
                   '5 TONNE 2 WHEELER',
                   'SINGLE AXLE TWO WHELER SEMI TRAILER',
                   '5TON 2WHEEL SEMI TIPPING TRAILER IRON BODY BSIII',
                   '2TONS 2WHEELER SEMI TIPPING TRAILER',
                   '5TONS 2WH TRACTOR SEMI TRAILERTIPPINGIRONBODY',
                   '5TONS 2WH TRAILERS',
                   '5 TON 2WH SEMI TRAILER WITH TIPPING',
                   '5 tons 2 Wheeler semi TIPPING Trailer',
                   '5 TONS OF 2 WHEELER TIPPING TRAILER',
                   '5TONN 2WH TRAILERS',
                   '5 TONS 2WHEEL SEMI TIPPING TRAILER',
                   '5 TONNS 2WH TRAILER',
                   '5TONSSINGLE AXLE2 WHEL SCB TIPPING TRAILER',
                   '5 TONS 2 WHEELER SEMI TIPPING TRAILER BSIII',
                   '5TONS 2 WHEELER SEMI TIPPIG TRAILER',
                   '5 TON 2 WHEELER SEMI TRAILER TIPPING',
                   '3 TON 2 WH SEMI TIPPING TRAILER IRON BODY',
                   '5 TONS 2WHEELERSEMITIPPING TRATILERIRONB',
                   '5TON 2WHEELER SEMI TIPPING TRAILER',
                   '6.15 TONS 2 WHEELER SEMI TIPPING IRON BODY',
                   '5TONS2WHEELER SEMI TIPPINGTRAILER IRONBODY',
                   '5TON 2WHEELERS SEMI TIPPING TRAILER WORKS',
                   '5TONS 2 WHEELER SEMI TIPPING TRAILER',
                   '5 TONNS 2WH TRAILERS',
                   '5 TON 2 WHEELER TIPPING TRAILER IRON BODY',
                   '5 TONS 2WHLRSEMITIPPING TRAILERIRON BODY',
                   '5TON 2WHELEER SEMI TIPPING TRAILER',
                   'TRACTORTRAILER  5TONS (2WHEELER)',
                   '5 TON 2 WH SEMI TRAILER TIPPING IRON BODY',
                   '3TON 2WHEEL SEMI TIPPING TRAILER IRON BODY',
                   '5 TON 2 WH.SEMI-TRAILOR TIPPING',
                   '5 TONS 2 WHEELER TIPPING  SEMI TRAILER',
                   '5TONNS 2WH SEMI TRAILER TIPPING IRON BODY BSIV',
                   '5 TONS 2 WHEELER SEMI TRAILER TIPPING',
                   '5 TONNS 2 WH TRAILERS',
                   '5 TONS 2WHELEER SEMI TIPPING TRAILER',
                   '5 TONS 2 WHEELER TIPPING TRIALER',
                   '5TON 2WHEELER SEMI TRAILER TIPPING',
                   'TACTOR 2WHEELER SEMITRAILOR TIPPING - 5 TONNS',
                   'TWO WHEELER TRAILOR TIPPING - 5 TONNS',
                   '3 TON 2 WHEELER SEMI TIPPING TRAILER',
                   '5 TONS 2 WHEELR TIPPING TRAILER',
                   '5 TONS 2 WHEELER SEMI TIPPING TRAILER',
                   '5TONS2WHEELER SEMI TIPPINGTRAILER IRONBODY BSIII',
                   'TWO WHEELER TRACTOR TRAILOR - 5 TONNS',
                   'TWO WHEELER TRAILOR - 5 TONNS',
                   '5 TONS 2 WHEELER TIPPING SEMI  TRAILER',
                   '5 TONNS 2WH TIPPING TRAILERS',
                   '5TON 2WHEEL SEMI TIPPING TRAILER IRON BODY',
                   '5TONNS 2WH TRAILERS',
                   '3 TONS 2 WHEELER SEMI TIPPING T IRON BODY',
                   '5TONNES 2WH TIPPING TRAILERS',
                   '5TONNS 2WH SEMI TRAILER TIPPING',
                   '5TON 2WH TIPPING TRAILER',
                   '5 TONNS 2WH TIPPING TYPE TRAILERS',
                   'TWO WHEELER  TRACTOR TRAILOR - 5 TONNS',
                   '3TONS 2WHEELER TIPPING TRAILER IRON BODY',
                   '3TONS 2WHEELR SEMI TIPPING TRAILER',
                   '3TONS OF 2WHEELER SEMI TIPPING TRAILR',
                   '5TONS 2WH SEMI TRAILER TIPPPING IRON BODY',
                   '5TONS 2WHEELER SEMI TIPPING TRAILER',
                   '5TON 2 WHEELER SEMI TIPPING TRAILER',
                   '5TONN 2WH TIPPING TRACTOR TRAILER',
                   '5 TON 2 WHEELER SEMI TR. TIPPING',
                   '5 TONS 2WHEELER TIPPING TRAILER',
                   '5TONS 2WH SEMI TRAILER TIPPING IRON BODY',
                   '5TONNS 2WH SEMI TRAILER TIPPING IRON BODY',
                   '2 TONS 2 WHEELER SEMI TIPPING T IRON BODY',
                   '5TON 2WH SEMI TRAILER TIPPING IRON BODY',
                   '5TONNES 2WH TRAILERS TIPPING',
                   '5 TONS 2 WHEELER SEMI TIPPING  IRON BODY',
                   '5TON 2WH TRACTOR TRAILER WITH TIPPING',
                   '5 TONS 2 WHEELER TIPPING SEMI TRAILER',
                   '2.5 TONS OF 2 WHEELER TRAILER',
                   'SR2-R2-2AX SEMI TRAILER',
                   '5 TONS TWO WHEELER SEMI TIPPING TRAILER',
                   '5 TONS 2 WHEELER SEMI TRAILER TIPPING IRON BODY',
                   '5TONS 2 WHEELER TIPPING TRAILER',
                   '5 TONS OF 2 WHEELER SEMI TRAILOR TIPPING IRON BODY',
                   'TWO WHEELER TRACTOR TRAILOR TIPPING - 5 TONNS',
                   '6.25 MT(GVW) SINGLE AXLE TIPPING TRAILER',
                   '5TONNS 2WH TIPPING TRAILERS',
                   '5 TONS 2 WHEELER SEMI TIPPING IRON BODY',
                   '5 TON 2WH SEMI TRAILER TIPPING',
                   '5 TONS 2WH  SEMI TRAILER TIPPING IRON BODY']

water_trailer = ['5 TONS 2 WHEELER WATER TANKER T IRON BODY',
                 '5 TONS 2 WHEELER WATER TANKER  IRON BODY',
                 '5TONS 2WHEELER WATER TANKER',
                 '5 TONNES 2 WHEELER (WATER TANKER)',
                 '3 TONS 2 WHEELER WATER TANKER T IRON BODY',
                 'WATER TANKER  5TONS  2WHEELER TRAILER',
                 '5 TON 2 WHEELER WATER TANKER IRON BODY',
                 '5TONS 2WH SEMI TRAILER TIPPING WATER TANKER',
                 '5 TONS 2WHEEL WATER TANKER',
                 '5 TONS 2 WHEELER WATER TANKER',
                 '5TON 2WHEELER WATER TANKER',
                 '5 TONNS 2WH SEMI TRAILER WATER TANK',
                 '5TONS2WHEELER WATERTRANK TRAILERIRONBODY',
                 '3 TONS 2 WHEELER WATER TANKER  IRON BODY',
                 '5T 2WH SEMI TRAILER WATER TANKER IRON BODY',
                 '5TON 2WH WATER TANKER IRON BODY',
                 '5TONS 2WH TRAILER WATER TANKER IRON BODY',
                 '5TONS 2WH WATER TANKER IRON BODY',
                 '5 TONS 2WHEELER WATER TANKER',
                 '5TONS 2 WHEELER WATER TANKER T IRON BODY',
                 '5TON 2WH SEMI TRAILER(WATER TANKER)',
                 'WATER TANKER 5TONS 2WHEELER TRAILER BSIV',
                 '5 TONS OF 2 WHEELER WATER TANKER',
                 '5TONS 2 WHEELER WATER TANKER',
                 'WATER TANKER 5TONS 2WHEELER TRAILER',
                 '5TONNS 2WHEELER WATER TANKER',
                 '5 TON 2 WH TRAILER WATER TANKER TIPPING IRON BODY',
                 '3TONS 2 WHEELER WATER TANKER T IRON BODY',
                 '5TONNS 2WH WATER TANKER TRAILER IRON BODY',
                 '5TONS 2WHEELER WATER TANKER IRON BODY',
                 '5TON 2WH SEMI TRAILER IRON BODY(WATER TANKER)',
                 '5 TONS 2 WHEELER WATER TANKER IRON BODY',
                 '5TONS 2WHEELER WANTER TANKER',
                 '5 TONS 2WH WATER TANKER IRON BODY',
                 '5TONS SINGLE AXLE 2WTIPT5000L WATER TANKER']

fourWeelTrailer = ['10 TON 4WHEEL SINGLE AXLE SEMI TRAILER',
                   'DI-42RXHDM4WD4WHEEL4WHEELDRIVENGPATRACTOR BSIIIA',
                   '10TONNS 4WH TIPPER TRAILERS',
                   '10TONNES 4WH TRAILER',
                   '10TONS4WHEELER SINGLEAXLEEMITRAILERIRONBOY',
                   '10TON 4WHEELERSINGLEAXLESEMITIPPINGTRAILER',
                   '12 TON 4WHEELER NON-TIPPING TRIALER',
                   '10TONS 4WHEELER SINGLE AXLE TRAILER',
                   '10TONS 4 WHEELER SINGLE AXLE SEMI TTRAILR',
                   '10TONS 4WH SEMI TRAILER IRON BODY',
                   '10 TONNS 4 WH TRACTOR TRAILER',
                   '12 TONS 4 WHEELER NON-TIPPING TRAILLER IB',
                   '10TONS 4WHEELER SEMISINGLE AXLE FLATBED OP',
                   '10 TONS FOUR WHEELER TRAILOR',
                   '10 TONS FOUR WHEELER TRACTOR  TRAILOR',
                   '10 TONS 4 WHEELER SINGLE AXLE FLAT BED',
                   '12 TONS FOUR WHEELER TRAILOR',
                   '10TONS 4 WHEELER SINGLE AXLE TRAILER',
                   '10TONS 4WHEELER SINGLE AXLE SEMI-TRAILER',
                   '10TONS 4 WHEELER SINGLE AXLE SEMI TTIOB BSIV',
                   '12 TONS 4 WHEELER NON TIPPING TRAILER']

multiAxleTrailer = ['49MT TRIPLE AXLE SEMI TRAILER IRON BODY',
                    '35.02 MT TANDEM AXLED SEMI TRAILER IRON BODY']

for value in tqdm(tractor_trailer):
    rta_DF.loc[rta_DF.Model_Desc == value, 'Fuel'] = 'Non_Fuel'
    
for value in tqdm(water_trailer):
    rta_DF.loc[rta_DF.Model_Desc == value, 'Fuel'] = 'Non_Fuel'

for value in tqdm(fourWeelTrailer):
    rta_DF.loc[rta_DF.Model_Desc == value, 'Fuel'] = 'Non_Fuel'

for value in tqdm(multiAxleTrailer):
    rta_DF.loc[rta_DF.Model_Desc == value, 'Fuel'] = 'Non_Fuel'

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.30it/s]


In [42]:
print(f'Number of data points in the  Registration Dataset\t: {rta_DF.shape[0]}')
print(f'Number of missing values in the revised dataset\t\t: {sum(rta_DF.isnull().sum())}')

Number of data points in the  Registration Dataset	: 9608322
Number of missing values in the revised dataset		: 0


#### 4.5 Type-Casting to correct datatype

In [43]:
# Type-Casting to correct datatype

rta_DF.District = rta_DF.District.astype('string')
rta_DF.Model_Desc = rta_DF.Model_Desc.astype('string')
rta_DF.vehicleClass = rta_DF.vehicleClass.astype('string')
rta_DF.Fuel = rta_DF.Fuel.astype('string')
rta_DF.SeatingCapacity = rta_DF.SeatingCapacity.astype('int')
rta_DF.TempRegnNo = rta_DF.TempRegnNo.astype('string')
rta_DF.Category = rta_DF.Category.astype('string')
rta_DF.SecondVehicle = rta_DF.SecondVehicle.astype('string')
rta_DF.Manufacturer_Name = rta_DF.Manufacturer_Name.astype('string')

display(rta_DF.info())
rta_DF.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9608322 entries, 0 to 9608321
Data columns (total 11 columns):
 #   Column             Dtype         
---  ------             -----         
 0   District           string        
 1   Model_Desc         string        
 2   vehicleClass       string        
 3   Fuel               string        
 4   SeatingCapacity    int64         
 5   fromdate           datetime64[ns]
 6   TempRegnNo         string        
 7   Category           string        
 8   SecondVehicle      string        
 9   makeYear           datetime64[ns]
 10  Manufacturer_Name  string        
dtypes: datetime64[ns](2), int64(1), string(8)
memory usage: 806.4 MB


None

,District,Model_Desc,vehicleClass,Fuel,SeatingCapacity,fromdate,TempRegnNo,Category,SecondVehicle,makeYear,Manufacturer_Name
0,Adilabad,ACTIVA 5GWEAS&KS&DCBS(CBS)WSEHEETWHEEL BSIV,Motor Cycle,Petrol,2,2019-01-01,TS01LTR7367,Non Transport,N,2018-01-11,HONDA MOTORCYCLE&SCOOTER(I)P L
1,Adilabad,MAHINDRA XUV5OO FWD W11 BSIV,Motor Car,Diesel,7,2019-01-01,TS16XTR5445,Non Transport,Y,2018-01-08,M/S MAHINDRA &MAHINDRA LTD
2,Adilabad,LIVO WK S &S S W F D &R D B W ALLOYWHEELS BSIV,Motor Cycle,Petrol,2,2019-01-01,TS01LTR7329,Non Transport,N,2018-01-06,HONDA MOTORCYCLE&SCOOTER(I)P L
3,Adilabad,"""PASSION PRO""(I3S-SELF-DRUM-CAST) BSIV",Motor Cycle,Petrol,2,2019-01-01,TS01LTR7424,Non Transport,N,2018-01-12,HERO MOTOCORP LTD
4,Adilabad,CB SHINE W F DRUM B K S&E S W CAST WHEELS BSIV,Motor Cycle,Petrol,2,2019-01-01,TS01LTR7374,Non Transport,N,2018-01-11,HONDA MOTORCYCLE&SCOOTER(I)P L


#### 4.6 Saving RTA Dataset

In [44]:
# Saving DataFrame to Parquet : Making parts to reduce file size for github

chunk_size = rta_DF.shape[0]//3
for idx in range(0, len(rta_DF), chunk_size):
    RTA_Reg_path = f'{interim_file_path}/RTA_Reg_Data_P{idx // chunk_size + 1}.parquet'
    if not os.path.isfile(RTA_Reg_path):
        rta_DF.iloc[idx : idx + chunk_size].to_parquet(RTA_Reg_path, engine = 'pyarrow')